In [1]:
import pandas as pd
from datetime import datetime
from pandas_gbq.auth import get_credentials
import warnings
from pathlib import Path
from google.cloud import bigquery
from google.api_core.exceptions import NotFound, GoogleAPICallError
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
pd.set_option('display.max_columns', None)

c:\Users\xt11442\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.6) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\xt11442\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.6) which Google will stop supporting in new releases of google.cloud.bigquery_storage_v1 once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.cloud.bigquery_storage_v1 past that date.
  warnings.warn(message, FutureWarning)
c:\Users\xt11442\AppData\Loc

In [2]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
FOLDER_PATH= "C:/DATOS/Emision mes regular/"
DATASET_ID = "produccion"
TABLE_ID= "Reporte_Emision_mes_regular"
TABLE_CONTROL_ID= "CONTROL_reporte_emision_MR"

In [31]:
carpeta= Path(FOLDER_PATH)
columnas = ["PRIMER_IDELOTE", "NUMCERTIFICADOEXT", "MES_DECLARACION", "ANIO_DECLARACION", "VIGENCIA_INICIO",
    "VIGENCIA_FIN", "MONEDA", "PRIMA", "PROCESO", "ESTADO", "IDETRANSFERENCIA", "IDEDET" ]

# Autenticarse en BIGQUERY
credentials, _ = get_credentials()

bigquery_client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

In [39]:
def cargar_archivo(archivo):
    df = pd.read_csv(archivo, header=None, skiprows=1, dtype=str)
    if df.shape[1] == 13:
        # Tiene la numeración adicional
        df = df.iloc[:, 1:]
    elif df.shape[1] == 12:
        # No tiene numeración, no eliminamos nada
        pass
    else:
        raise ValueError(f"Estructura inesperada: {df.shape[1]} columnas en {archivo}")
    df.columns = columnas
    return df

def buscar_archivo(filename_txt: str) -> bool:
    # Verificar si existe la tabla de control
    table_control_ref = bigquery_client.dataset(DATASET_ID).table(TABLE_CONTROL_ID)
    if not tabla_existe(table_control_ref):
        return False
    else:
        # Buscar archivo en la tabla de control
        query = f"""
            SELECT COUNT(*) AS count
            FROM `{table_control_ref}`
            WHERE ARCHIVO_TXT = @archivo_txt
        """
        job_config = bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("archivo_txt", "STRING", filename_txt)]
        )

        df = bigquery_client.query(query, job_config=job_config).to_dataframe()
        return df["count"].iloc[0] > 0


def tabla_existe(table_ref):
    try:
      tabla = bigquery_client.get_table(table_ref)
      return True
    except NotFound:
        # La tabla no existe
        return False
    except GoogleAPICallError as e:
        # Otros errores de BigQuery (permisos, conexión, etc.)
        print(f"⚠️ Error al consultar BigQuery: {e}")
        raise


#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id):
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    job_config = bigquery.LoadJobConfig()
    if tabla_existe(table_ref):
        # Abre la tabla para agregar registros
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND
    else:
        tabla_tramas = bigquery.Table(table_ref)
        tabla_tramas = bigquery_client.create_table(tabla_tramas)
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
        print(f'ℹ️ ----- Se ha creado la tabla: "{table_id}" en el dataset: "{dataset_id}" -----')
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    #print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return

def Procesar_File_txt(carpeta):
    for ruta in carpeta.iterdir():
        if ruta.is_dir():
            fecha = datetime.strptime(ruta.name, "%Y%m").date()
            print(f"⏳ ... PROCESANDO CARPETA: {ruta.name} ... ⏳")
            for archivo in ruta.rglob("*.xlsx"):
                file_name= archivo.name
                if not buscar_archivo(file_name):
                    print(f"⏳ ... PROCESANDO ARCHIVO: {archivo} ... ⏳")
                    #df_reporte = cargar_archivo(archivo)
                    df_reporte= pd.read_excel("C:/DATOS/Emision mes regular/202609/reporte eduardo/TKT - septiembre.xlsx", sheet_name='Sheet1', 
                            dtype={'NUMCERTIFICADOEXT': str, 'IDETRANSFERENCIA': str, 'IDEDET':str})
                    #df_reporte['MES_DECLARACION'] = pd.to_numeric(df_reporte['MES_DECLARACION'], errors='coerce').astype('int64')
                    #df_reporte['ANIO_DECLARACION'] = pd.to_numeric(df_reporte['ANIO_DECLARACION'], errors='coerce').astype('int64')
                    #df_reporte['PRIMA'] = pd.to_numeric(df_reporte['PRIMA'], errors='coerce').astype('float64')
                    df_reporte['VIGENCIA_INICIO']= pd.to_datetime(df_reporte['VIGENCIA_INICIO'], format='%d/%m/%Y', errors='coerce').dt.date
                    df_reporte['VIGENCIA_FIN']= pd.to_datetime(df_reporte['VIGENCIA_FIN'], format='%d/%m/%Y', errors='coerce').dt.date
                    df_reporte['FECHA_REPORTE'] = fecha
                    df_reporte['NOMBRE_ARCHIVO'] = file_name
                    fecha_carga = datetime.now()
                    df_control = pd.DataFrame([{"ARCHIVO_TXT": file_name, "FECHA_CARGA": fecha_carga}])
                    Guardar_en_BigQuery(df_reporte, DATASET_ID, TABLE_ID)
                    Guardar_en_BigQuery(df_control, DATASET_ID, TABLE_CONTROL_ID)
                    print(f"  ✅ ARCHIVO {file_name} CARGADO ...")
                else:
                    print(f"  ⚠️ EL ARCHIVO {file_name} YA FUE CARGADO ANTERIORMENTE ...")
    return

##################### PRINCIPAL #########################
carpeta= Path(FOLDER_PATH)
Procesar_File_txt(carpeta)

⏳ ... PROCESANDO CARPETA: 202502 ... ⏳
⏳ ... PROCESANDO CARPETA: 202503 ... ⏳
⏳ ... PROCESANDO CARPETA: 202504 ... ⏳
⏳ ... PROCESANDO CARPETA: 202505 ... ⏳
⏳ ... PROCESANDO CARPETA: 202506 ... ⏳
⏳ ... PROCESANDO CARPETA: 202507 ... ⏳
⏳ ... PROCESANDO CARPETA: 202508 ... ⏳
⏳ ... PROCESANDO CARPETA: 202509 ... ⏳
⏳ ... PROCESANDO CARPETA: 202510 ... ⏳
⏳ ... PROCESANDO CARPETA: 202511 ... ⏳
⏳ ... PROCESANDO CARPETA: 202512 ... ⏳
⏳ ... PROCESANDO CARPETA: 202601 ... ⏳
⏳ ... PROCESANDO CARPETA: 202602 ... ⏳
⏳ ... PROCESANDO CARPETA: 202603 ... ⏳
⏳ ... PROCESANDO CARPETA: 202604 ... ⏳
⏳ ... PROCESANDO CARPETA: 202605 ... ⏳
⏳ ... PROCESANDO CARPETA: 202606 ... ⏳
⏳ ... PROCESANDO CARPETA: 202607 ... ⏳
⏳ ... PROCESANDO CARPETA: 202608 ... ⏳
⏳ ... PROCESANDO CARPETA: 202609 ... ⏳
⏳ ... PROCESANDO ARCHIVO: C:\DATOS\Emision mes regular\202609\reporte eduardo\prestamos - Octubre.xlsx ... ⏳
  ✅ ARCHIVO prestamos - Octubre.xlsx CARGADO ...
⏳ ... PROCESANDO ARCHIVO: C:\DATOS\Emision mes regular\202609\

# TEST

In [22]:
columnas = ["PRIMER_IDELOTE", "NUMCERTIFICADOEXT", "MES_DECLARACION", "ANIO_DECLARACION", "VIGENCIA_INICIO",
    "VIGENCIA_FIN", "MONEDA", "PRIMA", "PROCESO", "ESTADO", "IDETRANSFERENCIA", "IDEDET" ]

In [23]:
df_reporte = pd.read_csv(
    'C:/DATOS/Emision mes regular/202501/SAS/TC/20100130204_0158001_20250201_004.csv',
    header=None,
    skiprows=1,
    dtype= str
)

if df_reporte.shape[1] == 13:
    # Tiene la numeración adicional
    df_reporte = df_reporte.iloc[:, 1:]

elif df_reporte.shape[1] == 12:
    # No tiene numeración, no eliminamos nada
    pass

else:
    raise ValueError(
        f"Estructura inesperada: {df_reporte.shape[1]} columnas en archivo"
    )

df_reporte.columns = columnas

In [24]:
df_reporte.head()

,PRIMER_IDELOTE,NUMCERTIFICADOEXT,MES_DECLARACION,ANIO_DECLARACION,VIGENCIA_INICIO,VIGENCIA_FIN,MONEDA,PRIMA,PROCESO,ESTADO,IDETRANSFERENCIA,IDEDET
0,1281682,00110002304000002115,02,2025,10/12/2024,10/01/2025,SOL,1.30000,REP,PRO,303146460,742050759
1,1281682,00110002304000069295,02,2025,10/12/2024,10/01/2025,SOL,3.13000,REP,PRO,303146500,742050760
2,1281682,00110002304000075244,02,2025,10/12/2024,10/01/2025,SOL,18.42000,REP,PRO,303146543,742050761
3,1281682,00110002304000089830,02,2025,10/12/2024,10/01/2025,SOL,6.18000,REP,PRO,303146583,742050762
4,1281682,00110002304000090065,02,2025,10/12/2024,10/01/2025,SOL,18.43000,REP,PRO,303146620,742050763


In [28]:
df_reporte['MES_DECLARACION'] = pd.to_numeric(df_reporte['MES_DECLARACION'], errors='coerce').astype('int64')
df_reporte['ANIO_DECLARACION'] = pd.to_numeric(df_reporte['ANIO_DECLARACION'], errors='coerce').astype('int64')
df_reporte['PRIMA'] = pd.to_numeric(df_reporte['PRIMA'], errors='coerce').astype('float64')
df_reporte['VIGENCIA_INICIO']= pd.to_datetime(df_reporte['VIGENCIA_INICIO'], format='%d/%m/%Y', errors='coerce').dt.date
df_reporte['VIGENCIA_FIN']= pd.to_datetime(df_reporte['VIGENCIA_FIN'], format='%d/%m/%Y', errors='coerce').dt.date


In [29]:
df_reporte.head()

,PRIMER_IDELOTE,NUMCERTIFICADOEXT,MES_DECLARACION,ANIO_DECLARACION,VIGENCIA_INICIO,VIGENCIA_FIN,MONEDA,PRIMA,PROCESO,ESTADO,IDETRANSFERENCIA,IDEDET
0,1281682,00110002304000002115,2,2025,2024-12-10,2025-01-10,SOL,1.30,REP,PRO,303146460,742050759
1,1281682,00110002304000069295,2,2025,2024-12-10,2025-01-10,SOL,3.13,REP,PRO,303146500,742050760
2,1281682,00110002304000075244,2,2025,2024-12-10,2025-01-10,SOL,18.42,REP,PRO,303146543,742050761
3,1281682,00110002304000089830,2,2025,2024-12-10,2025-01-10,SOL,6.18,REP,PRO,303146583,742050762
4,1281682,00110002304000090065,2,2025,2024-12-10,2025-01-10,SOL,18.43,REP,PRO,303146620,742050763


In [30]:
df_reporte.info()

<class 'pandas.DataFrame'>
RangeIndex: 49928 entries, 0 to 49927
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   PRIMER_IDELOTE     49928 non-null  str    
 1   NUMCERTIFICADOEXT  49928 non-null  str    
 2   MES_DECLARACION    49928 non-null  int64  
 3   ANIO_DECLARACION   49928 non-null  int64  
 4   VIGENCIA_INICIO    49928 non-null  object 
 5   VIGENCIA_FIN       49928 non-null  object 
 6   MONEDA             49928 non-null  str    
 7   PRIMA              49928 non-null  float64
 8   PROCESO            49928 non-null  str    
 9   ESTADO             49928 non-null  str    
 10  IDETRANSFERENCIA   49841 non-null  str    
 11  IDEDET             49928 non-null  str    
dtypes: float64(1), int64(2), object(2), str(7)
memory usage: 7.1+ MB


# EXCEL

In [33]:
df_reporte= pd.read_excel("C:/DATOS/Emision mes regular/202609/reporte eduardo/TKT - septiembre.xlsx", sheet_name='Sheet1', 
                            dtype={'NUMCERTIFICADOEXT': str, 'IDETRANSFERENCIA': str, 'IDEDET':str})

In [36]:
df_reporte['VIGENCIA_INICIO']= pd.to_datetime(df_reporte['VIGENCIA_INICIO'], format='%d/%m/%Y', errors='coerce').dt.date
df_reporte['VIGENCIA_FIN']= pd.to_datetime(df_reporte['VIGENCIA_FIN'], format='%d/%m/%Y', errors='coerce').dt.date

In [37]:
df_reporte.head()

,PRIMER_IDELOTE,NUMCERTIFICADOEXT,MES_DECLARACION,ANIO_DECLARACION,VIGENCIA_INICIO,VIGENCIA_FIN,MONEDA,PRIMA,PROCESO,ESTADO,IDETRANSFERENCIA,IDEDET
0,1436767,00110002304000171537,10,2026,2026-09-18,2026-10-20,SOL,5.93,REP,PRO,375736747,942440613
1,1436767,00110002314000134399,10,2026,2026-09-18,2026-10-20,SOL,63.97,REP,PRO,375736802,942440614
2,1435437,00110002314000141875,10,2026,2026-09-10,2026-10-09,USD,2.31,REP,PRO,375104308,940568873
3,1436767,00110002334000156899,10,2026,2026-09-18,2026-10-20,SOL,12.14,REP,PRO,375736859,942440615
4,1434320,00110002354000103701,10,2026,2026-08-31,2026-09-30,SOL,9.20,REP,PRO,374953453,939079423


In [38]:
df_reporte.info()

<class 'pandas.DataFrame'>
RangeIndex: 4733 entries, 0 to 4732
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   PRIMER_IDELOTE     4733 non-null   int64  
 1   NUMCERTIFICADOEXT  4733 non-null   str    
 2   MES_DECLARACION    4733 non-null   int64  
 3   ANIO_DECLARACION   4733 non-null   int64  
 4   VIGENCIA_INICIO    4733 non-null   object 
 5   VIGENCIA_FIN       4733 non-null   object 
 6   MONEDA             4733 non-null   str    
 7   PRIMA              4733 non-null   float64
 8   PROCESO            4733 non-null   str    
 9   ESTADO             4733 non-null   str    
 10  IDETRANSFERENCIA   4730 non-null   str    
 11  IDEDET             4733 non-null   str    
dtypes: float64(1), int64(3), object(2), str(6)
memory usage: 661.6+ KB
